# 📊 Github Code Analyst

## ⚙️ 설정
- 필요 SDK 설치
- 필요 라이브러리 불러오기
- 환경변수 가져오기

In [ ]:
# %pip install --upgrade pip # 필요 시
PYGITHUB_VERSION = "2.8.1"

%pip install PyGithub=={PYGITHUB_VERSION}

In [ ]:
# Import
import asyncio
import os
import logging
import json

from pathlib import Path
from datetime import datetime
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List

from github import Github
from github import GithubException
from github.Auth import Token
from github.Repository import Repository

from dotenv import load_dotenv


# 초기 설정

## env
load_dotenv(override=True)

## logger
logger_level = logging.INFO
formatter = logging.Formatter('%(asctime)s - %(levelname)s - [%(name)s] %(message)s')

logger = logging.getLogger("GithubCodeAnalyst")
logger.setLevel(logger_level)

if logger.hasHandlers():
    logger.handlers.clear()
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)


In [ ]:
# Abstract
class _GithubLoginRequest(ABC):
    """
    Github 로그인을 위한 인증 요청 DTO
    """
    pass

class _GithubAuthenticator(ABC):
    @abstractmethod
    def githubLogin(self, request: _GithubLoginRequest) -> Github:
        pass
    
# Implementation
@dataclass
class GithubTokenLoginRequest(_GithubLoginRequest):
    """
    Github 개인 토큰을 이용한 로그인 요청 DTO
    """
    token: str
    def __post_init__(self):
        if not self.token or not self.token.strip():
            raise ValueError("GitHub 토큰은 비어있거나 공백일 수 없습니다.")

class GithubTokenAuthenticator(_GithubAuthenticator):
    def githubLogin(self, request: GithubTokenLoginRequest) -> Github:
        """
        Github 개인 토큰을 통해 로그인 및 Github 객체 생성

    
        Args:
            request (GithubTokenLoginRequest): GitHub 토큰이 포함된 DTO 객체.
        Returns:
            Github: 인증이 완료되어 API 통신에 바로 사용할 수 있는 PyGithub 클라이언트 객체.
        Raises:
            Exception: 토큰이 유효하지 않거나, GitHub API 서버와 통신할 수 없는 경우 발생합니다. 
        """
        github_token = Token(request.token)
        github = Github(auth=github_token)
        try:
            user = github.get_user()
            logger.info("[GITHUB_LOGIN] Success to login | username : " + user.login)
            return github
        except Exception as e:
            logger.error(f"[GITHUB_LOGIN] Fail to authenticate user | {e}", exc_info=True)
            raise

In [ ]:
# 로그인 전략 선택 및 객체 생성

## 토큰 불러오기
token_env_name = "GITHUB_TOKEN"
token_value = os.getenv(token_env_name)

if not token_value:
    raise RuntimeError(f"[GITHUB_LOGIN] 환경 변수 '{token_env_name}'가 설정되지 않았습니다. 확인해주세요.")

## 객체 생성
login_request = GithubTokenLoginRequest(token_value)
authenticator = GithubTokenAuthenticator()

In [ ]:
# 로그인
github = authenticator.githubLogin(login_request)

## 📦 데이터 수집 및 전처리
- 선택 가능 Repository 가져오기
- 내용을 가져올 Repository 선택하기
- 가져온 데이터를 chunk로 분리하여 생성

In [ ]:
# 선택 가능한 Repository 들 가져오기
# 토큰 접근 가능 레포 중, 실제로 커밋을 남긴 레포만 수집합니다.

MAX_CONCURRENT_REQUESTS = 10
_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

def check_contributed(user, repo: Repository) -> tuple[Repository, bool]:
    """
    사용자가 해당 Repository에 커밋을 남긴 적 있는지 확인합니다.

    get_contributors() 대신 get_commits(author=)를 사용하여
    전체 기여자 목록 조회 없이 빠르게 판별합니다.

    Args:
        user: 인증된 GitHub 사용자 객체
        repo (Repository): 확인할 Repository

    Returns:
        tuple[Repository, bool]: (레포, 기여 여부)
    """
    try:
        commits = repo.get_commits(author=user.login)
        is_contributor = commits.totalCount > 0
    except GithubException:
        # 빈 레포 등 커밋 조회 불가한 경우 False 처리
        is_contributor = False
    return repo, is_contributor

async def fetch_contributed_repo(user, repo: Repository) -> str | None:
    """
    사용자가 기여한 Repository의 full_name을 비동기로 가져옵니다.

    Args:
        user: 인증된 GitHub 사용자 객체
        repo (Repository): 확인할 Repository

    Returns:
        str | None: 'owner/repo_name' 형식, 미기여 또는 오류 시 None
    """
    async with _semaphore:
        try:
            repo_obj, is_contributor = await asyncio.to_thread(check_contributed, user, repo)
            return repo_obj.full_name if is_contributor else None
        except GithubException as e:
            logger.warning(f"[REPO_FETCH] 레포 접근 실패, 건너뜀 | {e}")
            return None
        except Exception as e:
            logger.error(f"[REPO_FETCH] 예상치 못한 오류 | {e}", exc_info=True)
            return None


In [ ]:
# 토큰 권한 내 모든 레포 가져오기 (조직 포함)
user = github.get_user()
repos = user.get_repos(type="all")  # public, private, fork, org 레포 포함

# 비동기 태스크 생성 및 실행
tasks = [fetch_contributed_repo(user, repo) for repo in repos]
results: List[str | None] = await asyncio.gather(*tasks)

# None 필터링 후 최종 목록 구성
my_repositories: List[str] = [name for name in results if name is not None]

logger.info(f"[REPO_FETCH] 총 {len(my_repositories)}개의 기여 Repository를 가져왔습니다.")